In [0]:
from datetime import datetime, timezone
from pathlib import Path
import json

# Project identity included in every Bronze record.
PROJECT_NAME = "trichy_east_election_pipeline"

# Unity Catalog locations.
CATALOG_NAME = "workspace"
BRONZE_SCHEMA_NAME = "bronze"

# Location of the original election source files.
BASE_VOLUME_PATH = (
    "/Volumes/workspace/elections/trichy_east"
)

# This notebook processes only candidate-list sources.
SOURCE_TYPE = "candidate_list"

# Trichy East Assembly Constituency number.
CONSTITUENCY_NO = "141"
CONSTITUENCY_NAME = "Trichy East"

# A unique batch ID makes every pipeline run traceable.
BATCH_ID = (
    "candidate_list_"
    + datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
)

# Permanent raw Bronze Delta table.
CANDIDATE_LIST_BRONZE_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA_NAME}."
    "trichy_east_candidate_list_raw"
)

# Git-managed source configuration.
MANIFEST_PATH = Path(
    "config/source_manifest.json"
)

print("Project:", PROJECT_NAME)
print("Source type:", SOURCE_TYPE)
print("Batch ID:", BATCH_ID)
print("Target table:", CANDIDATE_LIST_BRONZE_TABLE)

In [0]:
# Load the source inventory maintained in Git.
with open(MANIFEST_PATH, "r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Select only candidate-list files.
candidate_list_source_rows = []

for source in source_manifest["sources"]:
    if source["source_type"] == SOURCE_TYPE:
        candidate_list_source_rows.append(
            {
                **source,

                # Build the full Databricks Volume path.
                "source_file_path": (
                    f"{BASE_VOLUME_PATH}/"
                    f"{source['relative_path']}"
                ),

                # Standardize the name used by our notebook.
                "source_file_name": source["file_name"],

                # Standardize the strategy name used by our code.
                "configured_extraction_strategy": (
                    source["extraction_strategy"]
                )
            }
        )


assert len(candidate_list_source_rows) == 3, (
    "Expected one candidate-list file for each year."
)


for source in sorted(
    candidate_list_source_rows,
    key=lambda row: row["election_year"]
):
    print(
        source["election_year"],
        source["source_file_name"],
        source["configured_extraction_strategy"]
    )

In [0]:
import os


# Validate files before starting extraction.
# This prevents a missing or misspelled path from failing later.
missing_candidate_list_files = []

for source in sorted(
    candidate_list_source_rows,
    key=lambda row: row["election_year"]
):
    source_path = source["source_file_path"]
    file_exists = os.path.isfile(source_path)

    if file_exists:
        file_size_bytes = os.path.getsize(source_path)

        print(
            source["election_year"],
            "AVAILABLE",
            source["source_file_name"],
            file_size_bytes,
            "bytes"
        )
    else:
        print(
            source["election_year"],
            "MISSING",
            source_path
        )

        missing_candidate_list_files.append(source_path)


assert not missing_candidate_list_files, (
    f"{len(missing_candidate_list_files)} "
    "candidate-list source file(s) are missing."
)

print("Candidate-list source validation: PASS")

In [0]:
# pypdf handles the text-based 2016 candidate list.
# PyMuPDF renders the scanned 2021 PDF into images.
# Tesseract reads candidate details from those images.
%pip install pypdf==6.10.0 PyMuPDF==1.26.7 pytesseract==0.3.13 Pillow==11.3.0

In [0]:
import csv
import io
import fitz
import pytesseract

from PIL import Image
from pypdf import PdfReader

print("Candidate-list extraction libraries: READY")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)


# This schema preserves source evidence without assuming that
# candidate-list layouts are identical across election years.
CANDIDATE_LIST_BRONZE_SCHEMA = StructType([
    # Project and source lineage.
    StructField("project_name", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField("source_format", StringType(), True),

    # Election identity.
    StructField("constituency_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("election_year", IntegerType(), False),

    # Position inside the original source.
    StructField("page_no", IntegerType(), True),
    StructField("row_no", IntegerType(), True),

    # Common candidate fields retained exactly as extracted.
    StructField("candidate_position_raw", StringType(), True),
    StructField("candidate_name_raw", StringType(), True),
    StructField("party_name_raw", StringType(), True),
    StructField("symbol_raw", StringType(), True),

    # Complete raw evidence and variable source fields.
    StructField("raw_row_text", StringType(), True),
    StructField("raw_fields_json", StringType(), True),

    # Extraction diagnostics for later investigation.
    StructField("extraction_metadata_json", StringType(), True),
    StructField("parse_status", StringType(), False),
    StructField("parse_error", StringType(), True),
    StructField("extraction_method", StringType(), False),

    # Batch lineage and processing timestamps.
    StructField("batch_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("extraction_timestamp", TimestampType(), False)
])


print(
    "Candidate-list Bronze columns:",
    len(CANDIDATE_LIST_BRONZE_SCHEMA.fields)
)

In [0]:
def clean_candidate_list_text(value):
    """
    Apply only minimal technical cleanup.

    Bronze must preserve the source wording, so this removes
    control characters without normalizing names or parties.
    """
    if value is None:
        return None

    cleaned_value = str(value).replace(
        "\x00",
        ""
    ).strip()

    return cleaned_value or None


def build_candidate_list_record(
    source,
    page_no,
    row_no,
    raw_row_text,
    extraction_method,
    parse_status,
    parse_error=None,
    candidate_position_raw=None,
    candidate_name_raw=None,
    party_name_raw=None,
    symbol_raw=None,
    raw_fields=None,
    extraction_metadata=None
):
    """
    Build one record matching the stable Bronze schema.
    """
    return {
        "project_name": PROJECT_NAME,
        "source_file_name": source["source_file_name"],
        "source_file_path": source["source_file_path"],
        "source_format": source.get("format"),

        "constituency_name": CONSTITUENCY_NAME,
        "constituency_no": CONSTITUENCY_NO,
        "election_year": int(source["election_year"]),

        "page_no": page_no,
        "row_no": row_no,

        "candidate_position_raw": clean_candidate_list_text(
            candidate_position_raw
        ),
        "candidate_name_raw": clean_candidate_list_text(
            candidate_name_raw
        ),
        "party_name_raw": clean_candidate_list_text(
            party_name_raw
        ),
        "symbol_raw": clean_candidate_list_text(
            symbol_raw
        ),

        "raw_row_text": clean_candidate_list_text(
            raw_row_text
        ),

        # JSON preserves fields that vary between source formats.
        "raw_fields_json": json.dumps(
            raw_fields,
            ensure_ascii=False
        ) if raw_fields is not None else None,

        "extraction_metadata_json": json.dumps(
            extraction_metadata,
            ensure_ascii=False
        ) if extraction_metadata is not None else None,

        "parse_status": parse_status,
        "parse_error": parse_error,
        "extraction_method": extraction_method,

        "batch_id": BATCH_ID,
        "ingestion_timestamp": None,
        "extraction_timestamp": datetime.now(
            timezone.utc
        )
    }


print("Candidate-list Bronze record helper: READY")

In [0]:
def extract_candidate_list_pdf_text(source):
    """
    Preserve every readable text line from a candidate-list PDF.

    Candidate identification is intentionally deferred because
    headers, names and party details may span multiple lines.
    """
    records = []

    try:
        reader = PdfReader(source["source_file_path"])

        for page_index, page in enumerate(
            reader.pages,
            start=1
        ):
            try:
                page_text = page.extract_text(
                    extraction_mode="layout"
                ) or ""

                extracted_lines = [
                    clean_candidate_list_text(line)
                    for line in page_text.splitlines()
                ]

                # Blank lines carry no source content.
                extracted_lines = [
                    line for line in extracted_lines if line
                ]

                if not extracted_lines:
                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=page_index,
                            row_no=None,
                            raw_row_text=None,
                            extraction_method="pypdf_layout_text",
                            parse_status="FAILED",
                            parse_error=(
                                "No text extracted from page."
                            ),
                            raw_fields=None,
                            extraction_metadata={
                                "pdf_page": page_index,
                                "configured_strategy": source[
                                    "configured_extraction_strategy"
                                ]
                            }
                        )
                    )
                    continue

                for row_no, line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=page_index,
                            row_no=row_no,
                            raw_row_text=line,
                            extraction_method="pypdf_layout_text",
                            parse_status="SUCCESS",
                            raw_fields={
                                "line_1": line
                            },
                            extraction_metadata={
                                "pdf_page": page_index,
                                "line_number": row_no,
                                "configured_strategy": source[
                                    "configured_extraction_strategy"
                                ]
                            }
                        )
                    )

            except Exception as page_error:
                # Retain a failure record instead of silently
                # losing an unreadable page.
                records.append(
                    build_candidate_list_record(
                        source=source,
                        page_no=page_index,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pypdf_layout_text",
                        parse_status="FAILED",
                        parse_error=str(page_error),
                        extraction_metadata={
                            "failure_scope": "page"
                        }
                    )
                )

    except Exception as file_error:
        records.append(
            build_candidate_list_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pypdf_layout_text",
                parse_status="FAILED",
                parse_error=str(file_error),
                extraction_metadata={
                    "failure_scope": "file"
                }
            )
        )

    return records


print("Candidate-list PDF text extractor: READY")

In [0]:
def extract_candidate_list_pdf_ocr(source):
    """
    Render each PDF page as an image and extract raw text using OCR.

    This preserves OCR output in Bronze without deciding which lines
    represent candidate names, parties, symbols or headings.
    """
    records = []
    document = None

    try:
        document = fitz.open(source["source_file_path"])

        for page_index in range(document.page_count):
            page_no = page_index + 1

            try:
                page = document.load_page(page_index)

                # Render at 300 DPI to improve recognition quality.
                pixmap = page.get_pixmap(
                    dpi=300,
                    alpha=False
                )

                page_image = Image.open(
                    io.BytesIO(pixmap.tobytes("png"))
                )

                ocr_text = pytesseract.image_to_string(
                    page_image,
                    config="--psm 6"
                )

                extracted_lines = [
                    clean_candidate_list_text(line)
                    for line in ocr_text.splitlines()
                ]

                extracted_lines = [
                    line for line in extracted_lines if line
                ]

                if not extracted_lines:
                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=page_no,
                            row_no=None,
                            raw_row_text=None,
                            extraction_method="pymupdf_tesseract_ocr",
                            parse_status="FAILED",
                            parse_error="No text extracted by OCR.",
                            extraction_metadata={
                                "pdf_page": page_no,
                                "render_dpi": 300,
                                "tesseract_psm": 6
                            }
                        )
                    )
                    continue

                for row_no, line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=page_no,
                            row_no=row_no,
                            raw_row_text=line,
                            extraction_method="pymupdf_tesseract_ocr",
                            parse_status="SUCCESS",
                            raw_fields={"line_1": line},
                            extraction_metadata={
                                "pdf_page": page_no,
                                "line_number": row_no,
                                "render_dpi": 300,
                                "tesseract_psm": 6
                            }
                        )
                    )

            except Exception as page_error:
                records.append(
                    build_candidate_list_record(
                        source=source,
                        page_no=page_no,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pymupdf_tesseract_ocr",
                        parse_status="FAILED",
                        parse_error=str(page_error),
                        extraction_metadata={
                            "failure_scope": "page"
                        }
                    )
                )

    except Exception as file_error:
        records.append(
            build_candidate_list_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pymupdf_tesseract_ocr",
                parse_status="FAILED",
                parse_error=str(file_error),
                extraction_metadata={
                    "failure_scope": "file"
                }
            )
        )

    finally:
        if document is not None:
            document.close()

    return records


print("Candidate-list OCR extractor: READY")

In [0]:
def extract_candidate_list_csv_raw(source):
    """
    Preserve every physical line from the 2026 CSV.

    The source is not a conventional rectangular CSV, so Bronze
    stores its physical lines and CSV parser output without trying
    to identify candidate columns.
    """
    records = []

    try:
        with open(
            source["source_file_path"],
            "r",
            encoding="utf-8-sig",
            errors="replace",
            newline=""
        ) as source_file:
            for row_no, physical_line in enumerate(
                source_file,
                start=1
            ):
                # Remove only newline characters.
                raw_line = physical_line.rstrip("\r\n")

                try:
                    parsed_fields = next(
                        csv.reader([raw_line])
                    )

                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=None,
                            row_no=row_no,
                            raw_row_text=raw_line,
                            extraction_method="csv_physical_line_raw",
                            parse_status="SUCCESS",
                            raw_fields={
                                "physical_line": raw_line,
                                "parsed_fields": parsed_fields,
                                "detected_field_count": len(
                                    parsed_fields
                                )
                            },
                            extraction_metadata={
                                "physical_line_number": row_no,
                                "configured_strategy": source[
                                    "configured_extraction_strategy"
                                ]
                            }
                        )
                    )

                except Exception as row_error:
                    # Preserve even malformed individual lines.
                    records.append(
                        build_candidate_list_record(
                            source=source,
                            page_no=None,
                            row_no=row_no,
                            raw_row_text=raw_line,
                            extraction_method="csv_physical_line_raw",
                            parse_status="FAILED",
                            parse_error=str(row_error),
                            raw_fields={
                                "physical_line": raw_line
                            },
                            extraction_metadata={
                                "failure_scope": "physical_line"
                            }
                        )
                    )

    except Exception as file_error:
        records.append(
            build_candidate_list_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="csv_physical_line_raw",
                parse_status="FAILED",
                parse_error=str(file_error),
                extraction_metadata={
                    "failure_scope": "file"
                }
            )
        )

    return records


print("Candidate-list raw CSV extractor: READY")

In [0]:
# Collect raw records from all three election years.
all_candidate_list_records = []


for source in sorted(
    candidate_list_source_rows,
    key=lambda row: row["election_year"]
):
    strategy = source[
        "configured_extraction_strategy"
    ].lower()

    source_format = source["format"].lower()

    print(
        f"Extracting {source['election_year']}: "
        f"{source['source_file_name']}"
    )
    print("Configured strategy:", strategy)


    # CSV candidate lists require physical-line preservation.
    if source_format == "csv":
        source_records = extract_candidate_list_csv_raw(
            source
        )

    # OCR strategies are used for scanned PDFs.
    elif "ocr" in strategy:
        source_records = extract_candidate_list_pdf_ocr(
            source
        )

    # Remaining PDFs use their embedded text layer.
    elif source_format == "pdf":
        source_records = extract_candidate_list_pdf_text(
            source
        )

    else:
        # Preserve unsupported sources as traceable failures.
        source_records = [
            build_candidate_list_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="unsupported_strategy",
                parse_status="FAILED",
                parse_error=(
                    f"Unsupported format or strategy: "
                    f"{source_format}, {strategy}"
                ),
                extraction_metadata={
                    "configured_strategy": strategy
                }
            )
        ]


    print("Records extracted:", len(source_records))

    all_candidate_list_records.extend(
        source_records
    )


# Convert the records into the stable Bronze schema.
candidate_list_bronze_df = (
    spark.createDataFrame(
        all_candidate_list_records,
        schema=CANDIDATE_LIST_BRONZE_SCHEMA
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)


# Review extraction coverage without displaying every raw line.
display(
    candidate_list_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "parse_status"
    )
)

print(
    "Total candidate-list Bronze records:",
    candidate_list_bronze_df.count()
)

In [0]:
from pyspark.sql.window import Window


# Confirm every configured source produced at least one successful record.
successful_source_count = (
    candidate_list_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .select("source_file_path")
    .distinct()
    .count()
)

print("Expected source files:", len(candidate_list_source_rows))
print("Successful source files:", successful_source_count)

assert successful_source_count == 3, (
    "At least one candidate-list source produced no successful records."
)


# Each source position should occur only once.
duplicate_positions_df = (
    candidate_list_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .groupBy(
        "source_file_path",
        "page_no",
        "row_no"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_position_count = duplicate_positions_df.count()

print("Duplicate source positions:", duplicate_position_count)

assert duplicate_position_count == 0, (
    "Duplicate candidate-list source positions were found."
)


# Display a small sample from every year for visual inspection.
sample_window = Window.partitionBy(
    "election_year"
).orderBy(
    F.col("page_no").asc_nulls_last(),
    "row_no"
)

candidate_list_sample_df = (
    candidate_list_bronze_df
    .filter(F.col("parse_status") == "SUCCESS")
    .withColumn(
        "sample_number",
        F.row_number().over(sample_window)
    )
    .filter(F.col("sample_number") <= 8)
    .select(
        "election_year",
        "page_no",
        "row_no",
        "raw_row_text",
        "extraction_method"
    )
    .orderBy(
        "election_year",
        F.col("page_no").asc_nulls_last(),
        "row_no"
    )
)

display(candidate_list_sample_df)

print("Candidate-list Bronze QA: PASS")

In [0]:
# Ensure the target Bronze schema exists.
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.bronze"
)


# Remove this batch before writing so rerunning the cell
# cannot create duplicate records.
if spark.catalog.tableExists(
    CANDIDATE_LIST_BRONZE_TABLE
):
    spark.sql(
        f"""
        DELETE FROM {CANDIDATE_LIST_BRONZE_TABLE}
        WHERE batch_id = '{BATCH_ID}'
        """
    )

    write_mode = "append"

else:
    # The first run creates the managed Delta table.
    write_mode = "overwrite"


(
    candidate_list_bronze_df
    .write
    .format("delta")
    .mode(write_mode)
    .saveAsTable(CANDIDATE_LIST_BRONZE_TABLE)
)


print(
    "Rows written:",
    candidate_list_bronze_df.count()
)
print("Target table:", CANDIDATE_LIST_BRONZE_TABLE)
print("Batch ID:", BATCH_ID)

In [0]:
# Read this batch back from Delta to verify the write.
written_candidate_list_df = (
    spark.table(CANDIDATE_LIST_BRONZE_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

expected_row_count = candidate_list_bronze_df.count()
written_row_count = written_candidate_list_df.count()

successful_source_count = (
    written_candidate_list_df
    .filter(F.col("parse_status") == "SUCCESS")
    .select("source_file_path")
    .distinct()
    .count()
)

print("Expected rows:", expected_row_count)
print("Rows read from Delta:", written_row_count)
print("Successful source files:", successful_source_count)

assert written_row_count == expected_row_count
assert successful_source_count == 3

display(
    written_candidate_list_df
    .groupBy(
        "election_year",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy("election_year")
)

print("Candidate-list Bronze write validation: PASS")